# break-midstream-audited — cut a runaway stream, and chain the cut

`on_exceed="break"` stops a streamed answer at the chunk where the allowance dies. You keep the partial output; the provider bills to the cut.

> **Offline.** No API key, no network — the provider is a fake with the real client's *shape*, or a
> committed cassette. This notebook runs in CI on Python 3.11 and 3.13 via `nbmake`, so if a cell
> below stops working the build goes red.
>
> Beside it, [`main.py`](main.py) is the same story as a script. The last cell here asserts what
> that script asserts.

In [ ]:
# The notebook sits beside the recipe, so its own module is importable. Everything below reuses the
# recipe's fixtures rather than re-inventing them — a notebook that built its own fake could drift
# away from what `main.py` proves and nobody would notice.
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd()))

In [ ]:
import main as recipe

recipe.main()

## What the offline run can and cannot prove

Offline, *"the provider stream was closed"* is asserted against the fake's own `close()` flag — an object this recipe owns. `LIVE=1` swaps in a real `OpenAI()` and asserts httpx's `response.is_closed` instead, which is the real claim.

⚠️ And a cap covers **both directions**: `budget(tokens=20)` counts the input too. The fake sends `messages=[]`, so offline all 20 tokens are available to the output, while a real ~12-token prompt leaves ~0 — which is why the live switch carries its own `LIVE_CAP_TOKENS`. A fake that sends no input hides that entirely.